In [ ]:
# ==============================================================================
# CÉLULA 1: ULA (Unidade Lógica e Aritmética)
# ==============================================================================

def limitar_8bits(valor):
    """Garante que qualquer valor permaneça estritamente no intervalo de 8 bits (0 a 255)."""
    return valor & 0xFF


def ula(operacao, a, b=0):
    a = limitar_8bits(a)
    b = limitar_8bits(b)

    if operacao == "ADD":
        resultado = a + b
    elif operacao == "SUB":
        resultado = a - b
    elif operacao == "AND":
        resultado = a & b
    elif operacao == "OR":
        resultado = a | b
    elif operacao == "XOR":
        resultado = a ^ b
    else:
        raise ValueError(f"Operação inválida: {operacao}")

    return limitar_8bits(resultado)

print("✓ Módulo da ULA carregado com sucesso!")


In [ ]:
# ==============================================================================
# CÉLULA 2: CPU SIMULADA (Fetch -> Decode -> Execute)
# ==============================================================================

class CPU:
    def __init__(self):
        # 4 registradores gerais de 8 bits
        self.registradores = {
            "R0": 0,
            "R1": 0,
            "R2": 0,
            "R3": 0
        }
        # Memória principal com 16 posições (0 a 15)
        self.memoria = [0] * 16
        # Program Counter (PC)
        self.pc = 0
        # Flag de parada
        self.halted = False

    def executar_instrucao(self, instrucao):
        partes = instrucao.replace(",", "").split()
        if not partes:
            return

        operacao = partes[0]

        # LOAD Rd, valor
        if operacao == "LOAD":
            registrador = partes[1]
            valor = int(partes[2])
            self.registradores[registrador] = limitar_8bits(valor)
            self.pc += 1

        # STORE Rd, endereco
        elif operacao == "STORE":
            registrador = partes[1]
            endereco = int(partes[2])
            if endereco < 0 or endereco >= len(self.memoria):
                raise ValueError(f"Endereço de memória inválido: {endereco}")
            self.memoria[endereco] = self.registradores[registrador]
            self.pc += 1

        # Operações da ULA: ADD, SUB, AND, OR, XOR
        elif operacao in ["ADD", "SUB", "AND", "OR", "XOR"]:
            reg_dest = partes[1]
            reg_orig = partes[2]
            val_a = self.registradores[reg_dest]
            val_b = self.registradores[reg_orig]
            self.registradores[reg_dest] = ula(operacao, val_a, val_b)
            self.pc += 1

        # JMP endereco
        elif operacao == "JMP":
            endereco = int(partes[1])
            if endereco < 0 or endereco >= len(self.memoria):
                raise ValueError("Endereço de salto inválido")
            self.pc = endereco

        # HALT
        elif operacao == "HALT":
            self.halted = True

        else:
            raise ValueError(f"Instrução desconhecida: {operacao}")

    def ciclo(self):
        if self.halted:
            return

        # FETCH
        instrucao = self.memoria[self.pc]
        print()
        print("--------------------------------")
        print("FETCH")
        print("PC:", self.pc)
        print("Instrução:", instrucao)

        # DECODE + EXECUTE
        print("DECODE + EXECUTE")
        self.executar_instrucao(instrucao)

        # ESTADO
        self.mostrar_estado()

    def executar(self, max_ciclos=100):
        ciclos = 0
        while not self.halted:
            if ciclos >= max_ciclos:
                raise RuntimeError("Limite máximo de ciclos atingido (loop infinito).")
            self.ciclo()
            ciclos += 1

    def mostrar_estado(self):
        print("Registradores:")
        for nome, valor in self.registradores.items():
            print(f"  {nome} = {valor:02X} ({valor})")

print("✓ Módulo da CPU carregado com sucesso!")


In [ ]:
# ==============================================================================
# CÉLULA 3: PROGRAMAS E DEMONSTRAÇÃO VISUAL
# ==============================================================================

# 1. Programa da Seção 7: Calcula 5 + 8 e salva em MEM[15]
print("==================================================")
print("       1. PROGRAMA DE EXEMPLO (5 + 8 = 13)")
print("==================================================")
cpu1 = CPU()
cpu1.memoria[0] = "LOAD R0, 5"
cpu1.memoria[1] = "LOAD R1, 8"
cpu1.memoria[2] = "ADD R0, R1"
cpu1.memoria[3] = "STORE R0, 15"
cpu1.memoria[4] = "HALT"

cpu1.executar()

print("\n" + "=" * 50)
print(f"Resultado final em MEM[15]: {cpu1.memoria[15]} (esperado: 13)")
print("=" * 50 + "\n\n")


# 2. Programa Próprio da Seção 11: 10 Instruções
# Expressão: ((20 + 15) - 10) XOR 7 = 30
print("==================================================")
print("       2. PROGRAMA PRÓPRIO (10 INSTRUÇÕES)")
print("==================================================")
cpu2 = CPU()
cpu2.memoria[0] = "LOAD R0, 20"     # R0 <- 20
cpu2.memoria[1] = "LOAD R1, 15"     # R1 <- 15
cpu2.memoria[2] = "ADD R0, R1"      # R0 <- 20 + 15 = 35
cpu2.memoria[3] = "STORE R0, 14"    # Salva resultado parcial em MEM[14]
cpu2.memoria[4] = "LOAD R2, 10"     # R2 <- 10
cpu2.memoria[5] = "SUB R0, R2"      # R0 <- 35 - 10 = 25
cpu2.memoria[6] = "LOAD R3, 7"      # R3 <- 7
cpu2.memoria[7] = "XOR R0, R3"      # R0 <- 25 XOR 7 = 30
cpu2.memoria[8] = "STORE R0, 15"    # Salva resultado final em MEM[15]
cpu2.memoria[9] = "HALT"            # Encerra execução

cpu2.executar()

print("\n" + "=" * 50)
print(f"Resultado parcial em MEM[14]: {cpu2.memoria[14]} (esperado: 35)")
print(f"Resultado final em MEM[15]:   {cpu2.memoria[15]} (esperado: 30)")
print("=" * 50)


In [ ]:
# ==============================================================================
# CÉLULA 4: BATERIA COMPLETA DE TESTES AUTOMATIZADOS (SEÇÃO 12)
# ==============================================================================

def teste_load():
    cpu = CPU()
    cpu.memoria[0] = "LOAD R0, 25"
    cpu.memoria[1] = "HALT"
    cpu.executar()
    assert cpu.registradores["R0"] == 25
    print("[OK] 1. LOAD")

def teste_store():
    cpu = CPU()
    cpu.memoria[0] = "LOAD R1, 77"
    cpu.memoria[1] = "STORE R1, 10"
    cpu.memoria[2] = "HALT"
    cpu.executar()
    assert cpu.memoria[10] == 77
    print("[OK] 2. STORE")

def teste_add():
    cpu = CPU()
    cpu.memoria[0] = "LOAD R0, 15"
    cpu.memoria[1] = "LOAD R1, 25"
    cpu.memoria[2] = "ADD R0, R1"
    cpu.memoria[3] = "HALT"
    cpu.executar()
    assert cpu.registradores["R0"] == 40
    print("[OK] 3. ADD")

def teste_sub():
    cpu = CPU()
    cpu.memoria[0] = "LOAD R0, 50"
    cpu.memoria[1] = "LOAD R1, 20"
    cpu.memoria[2] = "SUB R0, R1"
    cpu.memoria[3] = "HALT"
    cpu.executar()
    assert cpu.registradores["R0"] == 30
    print("[OK] 4. SUB")

def teste_logicas():
    # AND
    cpu_and = CPU()
    cpu_and.memoria[0] = "LOAD R0, 12"
    cpu_and.memoria[1] = "LOAD R1, 10"
    cpu_and.memoria[2] = "AND R0, R1"
    cpu_and.memoria[3] = "HALT"
    cpu_and.executar()
    assert cpu_and.registradores["R0"] == 8

    # OR
    cpu_or = CPU()
    cpu_or.memoria[0] = "LOAD R0, 12"
    cpu_or.memoria[1] = "LOAD R1, 10"
    cpu_or.memoria[2] = "OR R0, R1"
    cpu_or.memoria[3] = "HALT"
    cpu_or.executar()
    assert cpu_or.registradores["R0"] == 14

    # XOR
    cpu_xor = CPU()
    cpu_xor.memoria[0] = "LOAD R0, 12"
    cpu_xor.memoria[1] = "LOAD R1, 10"
    cpu_xor.memoria[2] = "XOR R0, R1"
    cpu_xor.memoria[3] = "HALT"
    cpu_xor.executar()
    assert cpu_xor.registradores["R0"] == 6

    print("[OK] 5. AND, OR e XOR")

def teste_avanco_pc():
    cpu = CPU()
    cpu.memoria[0] = "LOAD R0, 1"
    cpu.ciclo()
    assert cpu.pc == 1
    print("[OK] 6. Avanço correto do PC")

def teste_jmp():
    cpu = CPU()
    cpu.memoria[0] = "LOAD R0, 10"
    cpu.memoria[1] = "JMP 4"
    cpu.memoria[2] = "LOAD R0, 99"
    cpu.memoria[3] = "HALT"
    cpu.memoria[4] = "LOAD R1, 20"
    cpu.memoria[5] = "HALT"
    cpu.executar()
    assert cpu.registradores["R0"] == 10
    assert cpu.registradores["R1"] == 20
    print("[OK] 7. JMP")

def teste_halt():
    cpu = CPU()
    cpu.memoria[0] = "HALT"
    cpu.executar()
    assert cpu.halted is True
    print("[OK] 8. HALT")

def teste_programa_exemplo():
    cpu = CPU()
    cpu.memoria[0] = "LOAD R0, 5"
    cpu.memoria[1] = "LOAD R1, 8"
    cpu.memoria[2] = "ADD R0, R1"
    cpu.memoria[3] = "STORE R0, 15"
    cpu.memoria[4] = "HALT"
    cpu.executar()
    assert cpu.memoria[15] == 13
    print("[OK] 9. Programa de exemplo (MEM[15] == 13)")

def teste_limite_8bits():
    cpu = CPU()
    # Overflow em 8 bits: 250 + 10 = 260 -> 4
    cpu.memoria[0] = "LOAD R0, 250"
    cpu.memoria[1] = "LOAD R1, 10"
    cpu.memoria[2] = "ADD R0, R1"
    # Underflow em 8 bits: 5 - 10 = -5 -> 251
    cpu.memoria[3] = "LOAD R2, 5"
    cpu.memoria[4] = "SUB R2, R1"
    cpu.memoria[5] = "HALT"
    cpu.executar()
    assert cpu.registradores["R0"] == 4
    assert cpu.registradores["R2"] == 251
    print("[OK] 10. Representação estrita de 8 bits")

print("==================================================")
print("EXECUTANDO SUÍTE COMPLETA DE TESTES")
print("==================================================")
teste_load()
teste_store()
teste_add()
teste_sub()
teste_logicas()
teste_avanco_pc()
teste_jmp()
teste_halt()
teste_programa_exemplo()
teste_limite_8bits()
print("==================================================")
print("TODOS OS TESTES FORAM APROVADOS COM SUCESSO!")
print("==================================================")
